# CAMEL Ambient Solar Wind Validation

## Overview

This notebook demonstrates how to validate ambient solar wind models against in situ measurements near Earth. The workflow is based on the **CAMEL (Comprehensive Assessment of Models and Events using Library tools)** framework developed at the [Community Coordinated Modeling Center (CCMC)](https://ccmc.gsfc.nasa.gov/).

This effort is part of the international [ISWAT H1-01 Team](https://iswat-cospar.org/h1-01), which aims to establish a unified validation approach for solar wind models across the heliophysics and space weather community.

The validation procedures presented in this notebook series are also available through an interactive web application built with Plotly Dash and Python. The **Ambient Solar Wind Validation Dashboard** can be accessed here:

- [CCMC CAMEL Ambient Solar Wind Validation Dashboard](https://ccmc.gsfc.nasa.gov/camel/AmbientSolarWind/home)

These notebooks provide a reproducible version of the same validation workflow and are intended to help users explore, reproduce, and extend the validation procedures of the CCMC CAMEL dashboard.

**Notebook version:** 1.0

---

## Notebook Scope

### 01 API and Preprocessing

This notebook:
- retrieves solar wind observations and model outputs through the **CCMC CAMEL API**
- applies the preprocessing and alignment steps
- saves ready-to-use example datasets for the subsequent notebooks in this repository

---

## Notebook Series

This notebook is part of the following workflow:

1. `01_api_and_preprocessing.ipynb` — retrieve, preprocess, align, and save example datasets
2. `02_point_to_point_metrics.ipynb` — compute point-to-point metrics and visual diagnostics
3. `03_binary_event_metrics.ipynb` — compute threshold-based binary event metrics and visual diagnostics
4. `04_peak_analysis.ipynb` — compute peak-based event metrics and visual diagnostics
5. `05_multi_model_comparison.ipynb` — compare multiple model outputs side by side against in situ measurements

---

## References

- Reiss, M. A., et al. (2022), *Unifying the validation of ambient solar wind models*, **Advances in Space Research**. [Open access](https://www.sciencedirect.com/science/article/pii/S0273117722004057)
- Rastaetter, L., et al. (2019), *Comprehensive Assessment of Models and Events Using Library Tools (CAMEL) Framework: Time Series Comparisons*, **Space Weather**. [Open access](https://doi.org/10.1029/2018SW002043)

---

## Developers

- **Martin Reiss** — Community Coordinated Modeling Center, NASA Goddard Space Flight Center, USA
- **Maya Levisohn** — Community Coordinated Modeling Center, NASA Goddard Space Flight Center, USA

## Contact

- martin.a.reiss@outlook.com
- maya.b.levisohn@nasa.gov

---

### Imports and API helpers

The **CCMC CAMEL API** provides access to solar wind model outputs and in situ measurements. This makes it possible to fetch data directly into Python for analysis without manually downloading files.

⚠️ **Note on warnings**  
The API calls can create warnings that can be ignored:

```
InsecureRequestWarning: Unverified HTTPS request is being made to host 'ccmc.gsfc.nasa.gov'.
Adding certificate verification is strongly advised.
See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from pathlib import Path
import requests
import numpy as np
import pandas as pd

#The base url for the API is https://ccmc.gsfc.nasa.gov/camel-api/ 
ccmcURL = "https://ccmc.gsfc.nasa.gov/camel-api/"


### Return data as a JSON object

The getData function below retrieves data from the CAMEL API and returns it as a JSON object.

**Parameters**
- `studyID` *(str)* — identifier of the validation study (e.g., `"AmbientSolarWind"`)  
- `sourceID` *(str)* — model or data source ID (e.g., `"AmbSoWi-ML"` or `"OMNI_PT1H"`)  
- `locationID` *(str)* — location of measurements (e.g., `"L1"`)  
- `parameters` *(list of str)* — list of variables to request (e.g., `["Flow_Speed"]`)  
- `startTime` *(str, optional)* — start time in `YYYY-MM-DD` format  
- `endTime` *(str, optional)* — end time in `YYYY-MM-DD` format  
- `timeperiodID` *(str, optional)* — predefined time period identifier  

> Either a `timeperiodID` **or** both `startTime` and `endTime` must be provided.  

**Returns**  
- A JSON object containing the requested data.

In [ ]:
def getData(studyID, sourceID, locationID, parameters, startTime=None, endTime=None, timeperiodID=None):
    '''Returns a json file of the specified API call'''
    
    url = ccmcURL+'data?study='+studyID+'&source='+sourceID+'&location='+locationID

    #timeperiod can be identified either by start and end or by timePeriodID
    if startTime is not None and endTime is not None:
        timeURL = '&start='+startTime+'&end='+endTime
    elif timeperiodID is not None:
        timeURL = '&timeperiod='+timeperiodID
    else:
        timeURL = ''

    #the parameters are listed with commas in between 
    parametersURL = "&parameters="
    for name in parameters:
        parametersURL += name + ","
    parametersURL = parametersURL[:-1]

    #the full API call is strung together
    url += timeURL + parametersURL
    r = requests.get(url, verify=False).json()
    return r

### Return data as a Pandas DataFrame

The following helper method retrieves data from the CAMEL API and returns it as a **Pandas DataFrame**.  
It can be used directly or in combination with the JSON method above.

**Parameters**
- `studyID` *(str)* — identifier of the validation study (e.g., `"AmbientSolarWind"`)  
- `sourceID` *(str)* — model or data source ID (e.g., `"AmbSoWi-ML"` or `"OMNI_PT1H"`)  
- `locationID` *(str)* — location of measurements (e.g., `"L1"`)  
- `parameters` *(list of str)* — list of variables to request (e.g., `["Flow_Speed"]`)  
- `startTime` *(str, optional)* — start time in `YYYY-MM-DD` format  
- `endTime` *(str, optional)* — end time in `YYYY-MM-DD` format  
- `timeperiodID` *(str, optional)* — predefined time period identifier  

> Either a `timeperiodID` **or** both `startTime` and `endTime` must be provided.  

**Returns**  
- A Pandas `DataFrame` with:  
  - **first column** → timestamps  
  - **subsequent columns** → requested parameters

In [ ]:
def getDataFrame(studyID, sourceID, locationID, parameters, startTime=None, endTime=None, timeperiodID=None):
    '''Returns a pandas dataframe of the specified API call'''
    try:
        #uses helper method above to fetch data
        json_data = getData(studyID, sourceID, locationID, parameters, startTime, endTime, timeperiodID)
        df = pd.DataFrame(json_data['data'])
    except Exception as e:
        return None

    #label columns of dataframe
    df.columns = ['timestamp']+parameters
    
    return df

### Retrieve example datasets

This example uses the CAMEL time period `1963-01-TP-01` and the `Flow_Speed` parameter for:
- `OMNI_PT1H` observations at L1
- `AmbSoWi-ML`
- `Bravda`


In [ ]:
import matplotlib.pyplot as plt

# Load data
df_ambsowi_ml = getDataFrame('AmbientSolarWind', 'AmbSoWi-ML', 'L1', ['Flow_Speed'], timeperiodID='1963-01-TP-01')
df_omni = getDataFrame('AmbientSolarWind', 'OMNI_PT1H', 'L1', ['Flow_Speed'], timeperiodID='1963-01-TP-01')
df_bravda = getDataFrame('AmbientSolarWind', 'Bravda', 'L1', ['Flow_Speed'], timeperiodID='1963-01-TP-01')

# Save raw API outputs
raw_data_dir = Path("../example_data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

df_ambsowi_ml.to_csv(raw_data_dir / "ambsowi_raw.csv", index=False)
df_omni.to_csv(raw_data_dir / "omni_raw.csv", index=False)
df_bravda.to_csv(raw_data_dir / "bravda_raw.csv", index=False)

# Replace 9999.0 with NaN
df_omni.replace(9999.0, np.nan, inplace=True)
df_ambsowi_ml.replace(9999.0, np.nan, inplace=True)
df_bravda.replace(9999.0, np.nan, inplace=True)

# Convert 'timestamp' to datetime
df_omni['timestamp'] = pd.to_datetime(df_omni['timestamp'], errors='coerce')
df_ambsowi_ml['timestamp'] = pd.to_datetime(df_ambsowi_ml['timestamp'], errors='coerce')
df_bravda['timestamp'] = pd.to_datetime(df_bravda['timestamp'], errors='coerce')

print("AmbSoWi-ML rows:", len(df_ambsowi_ml))
print("OMNI rows:      ", len(df_omni))
print("BRaVDA rows:    ", len(df_bravda))
print("Raw files saved to:", raw_data_dir.resolve())

## Import preprocessing functions

Import the alignment and interpolation functions used to prepare the time series for the jupyter notebooks.

In [ ]:
import sys

# Allow imports from the repository root and utils folder
project_root = ".."
utils_path = "../utils"

if project_root not in sys.path:
    sys.path.append(project_root)
if utils_path not in sys.path:
    sys.path.append(utils_path)

from data_processing import align_and_interpolate

## Define the validation time window

This section defines the time interval used to create the processed example datasets. The Carrington Rotation time window is applied before the processed files are saved. The downstream notebooks therefore load these processed files directly, so they all use the same validation interval defined here. 

Note that the selected time interval here corresponds to the default in the CAMEL Ambient Solar Wind dashboard. 


In [ ]:
from sunpy.coordinates.sun import carrington_rotation_time
import astropy.units as u

cr_start = 2088
cr_end = 2105

cr_long_start = 360
cr_long_end = 1

start_date = pd.Timestamp(carrington_rotation_time(cr_start, cr_long_start * u.deg).datetime).tz_localize("UTC")
end_date = pd.Timestamp(carrington_rotation_time(cr_end, cr_long_end * u.deg).datetime).tz_localize("UTC")

print("Start:", start_date)
print("End:  ", end_date)


## Preprocess and align the time series

This section performs all preprocessing before any files are saved:
- replace fill values with `NaN`
- parse timestamps
- standardize OMNI timestamps to the start of the hour
- create the 27-day persistence baseline
- filter to the validation window
- coerce numeric values and drop invalid rows
- align all series to the OMNI time grid using linear interpolation


In [ ]:
for df in (df_omni, df_ambsowi_ml, df_bravda):
    df.replace(9999.0, np.nan, inplace=True)

for df in (df_omni, df_ambsowi_ml, df_bravda):
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

df_omni["Time"] = df_omni["timestamp"].dt.floor("H")

df_persistence_27 = df_omni.loc[:, ["Time", "Flow_Speed"]].copy()
df_persistence_27["Time"] = df_persistence_27["Time"] + pd.Timedelta(days=27)
df_persistence_27["Name"] = "Persistence (27-days)"

omni_ref = df_omni.loc[
    (df_omni["Time"] >= start_date) & (df_omni["Time"] <= end_date),
    ["Time", "Flow_Speed"]
].copy()

ambsowi_ml_pred = df_ambsowi_ml.loc[
    (df_ambsowi_ml["timestamp"] >= start_date) & (df_ambsowi_ml["timestamp"] <= end_date),
    ["timestamp", "Flow_Speed"]
].rename(columns={"timestamp": "Time"}).copy()

bravda_pred = df_bravda.loc[
    (df_bravda["timestamp"] >= start_date) & (df_bravda["timestamp"] <= end_date),
    ["timestamp", "Flow_Speed"]
].rename(columns={"timestamp": "Time"}).copy()

p27_window = df_persistence_27.loc[
    (df_persistence_27["Time"] >= start_date) & (df_persistence_27["Time"] <= end_date),
    ["Time", "Flow_Speed"]
].copy()

for d in (omni_ref, ambsowi_ml_pred, bravda_pred, p27_window):
    d["Flow_Speed"] = pd.to_numeric(d["Flow_Speed"], errors="coerce")
    d.dropna(subset=["Time", "Flow_Speed"], inplace=True)
    d.sort_values("Time", inplace=True)
    d.drop_duplicates(subset=["Time"], keep="first", inplace=True)

aligned_omni = align_and_interpolate(omni_ref, omni_ref, method="linear")
aligned_ambsowi = align_and_interpolate(ambsowi_ml_pred, omni_ref, method="linear")
aligned_bravda = align_and_interpolate(bravda_pred, omni_ref, method="linear")
aligned_p27 = align_and_interpolate(p27_window, omni_ref, method="linear")

## Prepare load-ready example datasets

The saved files are fully preprocessed and aligned, so later notebooks can load them directly.


In [ ]:
omni_obs = aligned_omni.set_index("Time")[["Flow_Speed"]]
ambsowi_mod = aligned_ambsowi.set_index("Time")[["Flow_Speed"]]
bravda_mod = aligned_bravda.set_index("Time")[["Flow_Speed"]]
p27_mod = aligned_p27.set_index("Time")[["Flow_Speed"]]

print("Observed shape:   ", omni_obs.shape)
print("AmbSoWi-ML shape: ", ambsowi_mod.shape)
print("Bravda shape:     ", bravda_mod.shape)
print("Persistence shape:", p27_mod.shape)

In [ ]:
output_dir = Path("../example_data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

omni_obs.to_csv(output_dir / "omni_obs.csv")
ambsowi_mod.to_csv(output_dir / "ambsowi_mod.csv")
bravda_mod.to_csv(output_dir / "bravda_mod.csv")
p27_mod.to_csv(output_dir / "p27_mod.csv")

print("Saved files to:", output_dir.resolve())